# Useful Hooks!

Two hooks that address common weaknesses in AI-assisted development on larger projects — both run automatically when Claude changes code and give immediate feedback. (Both live in the `queries` project's `hooks/` dir.)

## 1. TypeScript type-checking hook (`tsc.js`)

**The problem:** when Claude changes a function signature, it often misses some **call sites**. E.g. add a `verbose` param to a function in `schema.ts` — Claude updates the definition but misses the call in `main.ts`, leaving type errors it didn't catch.

**The hook (PostToolUse, after edits):**
- runs **`tsc --noEmit`** to type-check
- captures any errors
- **feeds the errors back to Claude** immediately
- prompts Claude to fix the other files

Works for any typed language with a type checker. For untyped languages, do the same with **automated tests** instead. It's lightweight and fast.

## 2. Query-duplication-prevention hook (`query_hook.js`)

**The problem:** in a big project with many DB queries, Claude sometimes **writes a duplicate** query instead of reusing an existing one — especially when queries are just one part of a larger task (e.g. "create a Slack integration that alerts on orders pending > 3 days" → it writes a new query instead of using the existing `getPendingOrders()`).

**The hook (a review loop):**
- triggers when Claude edits files in **`./queries`**
- **launches a separate Claude Code instance programmatically** (via the **Agent SDK**)
- that second instance **reviews the change** for similar existing queries
- if a duplicate is found, it **feeds feedback back** to the original instance
- the original is prompted to remove the duplicate and reuse the existing function

This is one Claude instance reviewing another's work — a code-review process built from AI.

## Implementation considerations

- Both use the **PreToolUse / PostToolUse** system.
- **tsc hook:** lightweight, runs quickly.
- **query hook:** heavier — it spins up a **separate Claude instance per review**, so it costs extra time + API usage.

| | Benefit | Cost |
|---|---------|------|
| Query hook | cleaner codebase, less duplication | extra time + API per `./queries` edit |

**Recommendation:** only monitor **critical directories** to keep overhead down. (These launch-a-Claude workflows use the **Agent SDK** — the next lesson.)

## Extending the idea

Broader principles you can reuse:
- Use **compiler/linter output** as immediate feedback to Claude.
- Implement **AI code review** with separate instances checking each other's work.
- **Focus monitoring** on high-value directories where consistency matters most.
- **Balance** automation benefits against performance/cost.

Find your workflow's specific pain points and write targeted hooks for them.

Next: **Another useful hook**.